# WTMM Calibration: Normalization Chain Verification

Trace every stage of the 2D WTMM pipeline against analytic test signals.
Line ridges |x - x0|^h with known h.

In [ ]:
# Cell 1: Imports + test signal generation

import numpy as np
import matplotlib.pyplot as plt
import mlx.core as mx
from numpy.fft import fftfreq
import sys

sys.path.insert(0, '/Users/amasaryk/projects/Creep/wavelet/XSmurfWrapper')
sys.path.insert(0, '/Users/amasaryk/projects/Creep/wavelet')

# --- CWT functions (from main notebook cell 8) ---
def compute_scales(n_oct, n_vox, a_min=1):
    norm = 6.0 / 0.86
    return [a_min * (2.0 ** (o + v / n_vox)) * norm
            for o in range(n_oct) for v in range(n_vox)]

def _build_wavelet_filters_f32(kx, ky, scale, wavelet='gaussian'):
    sx = kx * scale
    sy = ky * scale
    gauss = mx.exp(-(sx * sx + sy * sy))
    zero = mx.zeros_like(gauss)
    def to_complex(r, i):
        return mx.stack([r, i], axis=-1).view(mx.complex64).squeeze(-1)
    if wavelet == 'mexican':
        k2 = sx * sx + sy * sy
        return {
            'dx':  to_complex(zero, sx * k2 * gauss),
            'dy':  to_complex(zero, sy * k2 * gauss),
        }
    else:
        return {
            'dx':  to_complex(zero, sx * gauss),
            'dy':  to_complex(zero, sy * gauss),
        }

def cwt_mod_only(image, scales, pad=32, wavelet='gaussian'):
    ny, nx = image.shape
    padded = np.pad(image, pad, mode='reflect').astype(np.float32)
    ny_p, nx_p = padded.shape
    crop = (slice(pad, pad + ny), slice(pad, pad + nx))
    kx_np = fftfreq(nx_p).astype(np.float32)
    ky_np = fftfreq(ny_p).astype(np.float32)
    KX_np, KY_np = np.meshgrid(kx_np, ky_np)
    KX, KY = mx.array(KX_np), mx.array(KY_np)
    image_fft = mx.fft.fft2(mx.array(padded))
    n_sc = len(scales)
    mod = np.zeros((n_sc, ny, nx), dtype=np.float32)
    dx_out = np.zeros_like(mod)
    dy_out = np.zeros_like(mod)
    for i, scale in enumerate(scales):
        filters = _build_wavelet_filters_f32(KX, KY, scale, wavelet=wavelet)
        dx = np.array(mx.fft.ifft2(image_fft * filters['dx']).real)[crop]
        dy = np.array(mx.fft.ifft2(image_fft * filters['dy']).real)[crop]
        dx_out[i] = dx
        dy_out[i] = dy
        mod[i] = np.sqrt(dx * dx + dy * dy)
    mx.eval()
    return mod, dx_out, dy_out

# --- Test signals: 256x256, line ridges ---
N = 256
x = np.linspace(-1, 1, N)
X, Y = np.meshgrid(x, x)

test_signals = {}
h_values = [0.0, 0.2, 0.5, 0.8, 1.2]
positions = [-0.6, -0.3, 0.0, 0.3, 0.6]

for h_val, x0 in zip(h_values, positions):
    if h_val == 0.0:
        f = np.where(X > x0, 1.0, 0.0).astype(np.float64)
        label = f'step h=0 x={x0}'
    else:
        f = np.abs(X - x0) ** h_val
        label = f'|x-{x0}|^{h_val}'
    test_signals[label] = {'field': f, 'h_true': h_val, 'x0': x0}

# Smooth bump
f_smooth = np.exp(-20 * (X**2 + Y**2))
test_signals['smooth'] = {'field': f_smooth, 'h_true': np.inf, 'x0': 0.0}

# --- Plot ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, (label, info) in zip(axes.flat, test_signals.items()):
    ax.imshow(info['field'], cmap='gray', extent=[-1, 1, -1, 1])
    ax.axvline(info['x0'], color='red', ls='--', lw=1, alpha=0.7)
    ax.set_title(label, fontsize=10)
plt.suptitle('Test signals', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# Verify analytic h from cross-sections (log-log)
fig, ax = plt.subplots(figsize=(8, 6))
mid = N // 2
for label, info in test_signals.items():
    h_val = info['h_true']
    if h_val == 0 or h_val == np.inf: continue
    x0 = info['x0']
    ix0 = np.argmin(np.abs(x - x0))
    dx_arr = x[ix0+1:ix0+60] - x0
    df = np.abs(info['field'][mid, ix0+1:ix0+60] - info['field'][mid, ix0])
    valid = (dx_arr > 0) & (df > 0)
    if valid.sum() >= 3:
        p = np.polyfit(np.log10(dx_arr[valid]), np.log10(df[valid]), 1)
        ax.loglog(dx_arr[valid], df[valid], 'o', ms=3, label=f'{label}: slope={p[0]:.3f}')
        ax.loglog(dx_arr[valid], 10**p[1] * dx_arr[valid]**p[0], '--', lw=1)
ax.set_xlabel('|x - x0|'); ax.set_ylabel('|f(x) - f(x0)|')
ax.set_title('Analytic h verification (slope should match)')
ax.legend(fontsize=7); ax.grid(True, alpha=0.2, which='both')
plt.tight_layout(); plt.show()

print(f'{len(test_signals)} test signals generated')

## Cell 2: Raw CWT modulus at ridge locations -- THE CRITICAL TEST

Slope of log2(mod) vs log2(a) MUST equal h_true.
If not, the CWT normalization is wrong.

In [ ]:
# Cell 2: THE CRITICAL TEST -- CWT modulus scaling

scales_mapped = compute_scales(n_oct=4, n_vox=10)  # with 6.0/0.86
scales_raw = [2.0 ** (o + v / 10) for o in range(4) for v in range(10)]

print(f'Mapped scales: {len(scales_mapped)}, [{scales_mapped[0]:.2f}, {scales_mapped[-1]:.2f}] px')
print(f'Raw scales:    {len(scales_raw)}, [{scales_raw[0]:.2f}, {scales_raw[-1]:.2f}] px')
print(f'Scale factor: 6.0/0.86 = {6.0/0.86:.4f}')
print(f'NOTE: fftfreq returns cycles/sample (NOT 2*pi*cycles)')
print()

results_cwt = {}
mid = N // 2
window = 3  # search window around ridge for max mod

for wavelet_name in ['gaussian', 'mexican']:
    print(f'--- {wavelet_name} ---')
    for label, info in test_signals.items():
        if info['h_true'] == np.inf: continue
        x0 = info['x0']
        ix0 = np.argmin(np.abs(x - x0))

        mod_map, dx_map, dy_map = cwt_mod_only(info['field'], scales_mapped, wavelet=wavelet_name)
        mod_raw, _, _ = cwt_mod_only(info['field'], scales_raw, wavelet=wavelet_name)

        # Max mod in window around ridge at each scale
        mod_at_ridge_m = np.array([mod_map[si, mid, max(0,ix0-window):min(N,ix0+window+1)].max()
                                   for si in range(len(scales_mapped))])
        mod_at_ridge_r = np.array([mod_raw[si, mid, max(0,ix0-window):min(N,ix0+window+1)].max()
                                   for si in range(len(scales_raw))])
        dx_at = np.array([np.abs(dx_map[si, mid, max(0,ix0-window):min(N,ix0+window+1)]).max()
                          for si in range(len(scales_mapped))])
        dy_at = np.array([np.abs(dy_map[si, mid, max(0,ix0-window):min(N,ix0+window+1)]).max()
                          for si in range(len(scales_mapped))])

        results_cwt[(wavelet_name, label)] = {
            'h_true': info['h_true'], 'x0': x0,
            'mod_mapped': mod_at_ridge_m, 'mod_raw': mod_at_ridge_r,
            'scales_mapped': np.array(scales_mapped),
            'scales_raw': np.array(scales_raw),
            'dx_at': dx_at, 'dy_at': dy_at,
        }

# --- THE CRITICAL PLOT ---
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for col, wavelet_name in enumerate(['gaussian', 'mexican']):
    for row, (stype, skey, slabel) in enumerate([
        ('mapped', 'scales_mapped', 'MAPPED (with 6.0/0.86)'),
        ('raw', 'scales_raw', 'RAW (pure octave/voice)'),
    ]):
        ax = axes[row, col]
        for label, info in test_signals.items():
            if info['h_true'] == np.inf: continue
            key = (wavelet_name, label)
            r = results_cwt[key]
            sarr = r[skey]
            marr = r[f'mod_{stype}']
            log2_s = np.log2(sarr)
            log2_m = np.log2(np.maximum(marr, 1e-30))
            valid = marr > 1e-20

            # Fit in middle range
            lo = log2_s[valid].min() + 1
            hi = log2_s[valid].max() - 0.5
            fit_mask = valid & (log2_s > lo) & (log2_s < hi)
            if fit_mask.sum() >= 3:
                slope, intercept = np.polyfit(log2_s[fit_mask], log2_m[fit_mask], 1)
            else:
                slope = np.nan

            ax.plot(log2_s[valid], log2_m[valid], 'o-', ms=3, lw=1,
                    label=f'h={r["h_true"]:.1f}, slope={slope:.3f}')
            if fit_mask.sum() >= 3:
                ax.plot(log2_s[fit_mask], slope*log2_s[fit_mask]+intercept, '--', lw=2, alpha=0.5)

        ax.set_xlabel(f'log2(scale) [{slabel}]')
        ax.set_ylabel('log2(mod)')
        ax.set_title(f'{wavelet_name} / {slabel}')
        ax.legend(fontsize=6); ax.grid(True, alpha=0.2)

plt.suptitle('CRITICAL: Does mod scale as a^h?  Slope MUST = h_true',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# --- Summary table ---
print(f'\n{"="*75}')
print(f'  {"Wavelet":<10s} {"Signal":<20s} {"h_true":>7s} {"h_mapped":>10s} {"h_raw":>10s} {"error":>8s}')
print(f'{"="*75}')
for wn in ['gaussian', 'mexican']:
    for label, info in test_signals.items():
        if info['h_true'] == np.inf: continue
        key = (wn, label)
        r = results_cwt[key]
        h_vals = {}
        for stype in ['mapped', 'raw']:
            sarr = r[f'scales_{stype}']
            marr = r[f'mod_{stype}']
            log2_s = np.log2(sarr)
            log2_m = np.log2(np.maximum(marr, 1e-30))
            valid = marr > 1e-20
            lo = log2_s[valid].min() + 1
            hi = log2_s[valid].max() - 0.5
            fm = valid & (log2_s > lo) & (log2_s < hi)
            h_vals[stype] = np.polyfit(log2_s[fm], log2_m[fm], 1)[0] if fm.sum() >= 3 else np.nan
        err = h_vals['mapped'] - r['h_true']
        print(f'  {wn:<10s} {label:<20s} {r["h_true"]:>7.2f} {h_vals["mapped"]:>10.4f} {h_vals["raw"]:>10.4f} {err:>+8.4f}')
print(f'{"="*75}')
print(f'\nIf slopes != h_true, the CWT normalization is WRONG.')
print(f'If mapped != raw slope, the 6.0/0.86 factor affects scaling (it should not).')

## Cell 3: Gradient direction + spatial profile check

In [ ]:
# Cell 3: Gradient direction and spatial profile

test_label = '|x-0.0|^0.5'
test_info = test_signals[test_label]
ix0 = np.argmin(np.abs(x - test_info['x0']))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for col, wn in enumerate(['gaussian', 'mexican']):
    r = results_cwt[(wn, test_label)]
    ax = axes[0, col]
    ax.semilogy(np.log2(r['scales_mapped']), r['dx_at'], 'b.-', ms=3, label='|dx|')
    ax.semilogy(np.log2(r['scales_mapped']), r['dy_at'], 'r.-', ms=3, label='|dy|')
    ax.set_xlabel('log2(scale)'); ax.set_ylabel('magnitude')
    ax.set_title(f'{wn}: dx vs dy at ridge (dx should dominate)')
    ax.legend(); ax.grid(True, alpha=0.2)

# Spatial profiles at different scales
mod_full, _, _ = cwt_mod_only(test_info['field'], scales_mapped, wavelet='gaussian')
ax = axes[1, 0]
for si in [2, 10, 20, 30]:
    if si >= len(scales_mapped): continue
    ax.plot(x, mod_full[si, mid, :], lw=1, label=f'a={scales_mapped[si]:.1f}')
ax.axvline(test_info['x0'], color='red', ls='--')
ax.set_xlabel('x'); ax.set_ylabel('mod')
ax.set_title(f'Spatial profile of mod (gaussian)\n{test_label}')
ax.legend(fontsize=7); ax.grid(True, alpha=0.2)

# Does max position stay at x0?
ax = axes[1, 1]
ix_maxes = [np.argmax(mod_full[si, mid, :]) for si in range(len(scales_mapped))]
x_maxes = [x[ix] for ix in ix_maxes]
ax.plot(np.log2(scales_mapped), x_maxes, 'ko-', ms=3)
ax.axhline(test_info['x0'], color='red', ls='--', label=f'true x0={test_info["x0"]}')
ax.set_xlabel('log2(scale)'); ax.set_ylabel('x of max mod')
ax.set_title('Max mod position vs scale (should be constant)')
ax.legend(); ax.grid(True, alpha=0.2)

plt.suptitle(f'Gradient check: {test_label}', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## Cell 4: fftfreq normalization deep-dive

numpy.fft.fftfreq(N) returns cycles/sample.
The wavelet exp(-(kx*a)^2) uses these directly.
What is the actual real-space width?

In [ ]:
# Cell 4: What does the wavelet actually look like in real space?

from scipy.optimize import curve_fit

test_a = 10.0
N_test = 512
kx_1d = fftfreq(N_test).astype(np.float64)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for col, (wname, extra) in enumerate([
    ('gaussian', ''),
    ('mexican', ' (with k^2 factor)'),
]):
    # Build 1D wavelet filter
    if wname == 'mexican':
        filt_dx = 1j * kx_1d * test_a * (kx_1d * test_a)**2 * np.exp(-(kx_1d * test_a)**2)
    else:
        filt_dx = 1j * kx_1d * test_a * np.exp(-(kx_1d * test_a)**2)

    # Also the smoothing kernel (no derivative)
    filt_smooth = np.exp(-(kx_1d * test_a)**2)

    # Real space via IFFT
    wav_smooth = np.fft.ifftshift(np.fft.ifft(filt_smooth).real)
    wav_dx = np.fft.ifftshift(np.fft.ifft(filt_dx).real)
    xpx = np.arange(N_test) - N_test // 2

    # Top: smoothing kernel
    ax = axes[0, col]
    ax.plot(xpx, wav_smooth, 'k-', lw=1.5)
    ax.set_xlim(-5 * test_a, 5 * test_a)
    ax.set_title(f'{wname}: smoothing kernel (a={test_a}){extra}')
    ax.set_xlabel('pixels'); ax.grid(True, alpha=0.2)

    # Fit Gaussian to find actual width
    center = N_test // 2
    roi = slice(center - int(3*test_a), center + int(3*test_a))
    def gfunc(xx, A, sigma):
        return A * np.exp(-xx**2 / (2 * sigma**2))
    try:
        popt, _ = curve_fit(gfunc, xpx[roi], wav_smooth[roi], p0=[wav_smooth.max(), test_a])
        ax.plot(xpx[roi], gfunc(xpx[roi], *popt), 'r--', lw=1,
                label=f'Gaussian fit: sigma={popt[1]:.2f} px')
        ax.legend(fontsize=8)
        print(f'{wname} smoothing kernel: sigma_real = {popt[1]:.4f} px for scale param a = {test_a}')
        print(f'  ratio sigma/a = {popt[1]/test_a:.4f}')
        print(f'  If fftfreq gives cycles: sigma = a/(2*pi*sqrt(2)) = {test_a/(2*np.pi*np.sqrt(2)):.4f}')
        print(f'  If fftfreq gives radians: sigma = a/sqrt(2) = {test_a/np.sqrt(2):.4f}')
    except Exception as e:
        print(f'Fit failed: {e}')

    # Bottom: dx wavelet
    ax = axes[1, col]
    ax.plot(xpx, wav_dx, 'k-', lw=1.5)
    ax.set_xlim(-5 * test_a, 5 * test_a)
    ax.set_title(f'{wname}: dx wavelet (a={test_a})')
    ax.set_xlabel('pixels'); ax.grid(True, alpha=0.2)

    # Zero crossings
    sign_changes = np.where(np.diff(np.sign(wav_dx)))[0]
    if len(sign_changes) >= 2:
        support = xpx[sign_changes[-1]] - xpx[sign_changes[0]]
        print(f'  dx wavelet support (zero crossings): {support} px, support/a = {support/test_a:.2f}')

# Right column: compare different scales
ax = axes[0, 2]
for a_val in [5, 10, 20, 40]:
    filt = np.exp(-(kx_1d * a_val)**2)
    wav = np.fft.ifftshift(np.fft.ifft(filt).real)
    # Normalize to unit peak
    wav /= wav.max()
    ax.plot(xpx / a_val, wav, lw=1, label=f'a={a_val}')
ax.set_xlim(-5, 5)
ax.set_xlabel('x / a'); ax.set_ylabel('normalized amplitude')
ax.set_title('Smoothing kernels scaled by a\n(should collapse if self-similar)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.2)

ax = axes[1, 2]
for a_val in [5, 10, 20, 40]:
    filt = 1j * kx_1d * a_val * np.exp(-(kx_1d * a_val)**2)
    wav = np.fft.ifftshift(np.fft.ifft(filt).real)
    wav /= np.abs(wav).max()
    ax.plot(xpx / a_val, wav, lw=1, label=f'a={a_val}')
ax.set_xlim(-5, 5)
ax.set_xlabel('x / a'); ax.set_ylabel('normalized amplitude')
ax.set_title('dx wavelets scaled by a\n(should collapse)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.2)

plt.suptitle('Wavelet real-space shape: what does exp(-(k*a)^2) actually look like?',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## Cell 5: Diagnosis -- h_measured vs h_true

Fit h_meas = a * h_true + b.
- a != 1: multiplicative error
- b != 0: additive shift

In [ ]:
# Cell 5: Diagnosis

h_true_arr = []
h_meas_g = []
h_meas_m = []

for label, info in test_signals.items():
    if info['h_true'] == np.inf: continue
    h_true_arr.append(info['h_true'])
    for wn, hlist in [('gaussian', h_meas_g), ('mexican', h_meas_m)]:
        key = (wn, label)
        r = results_cwt[key]
        log2_s = np.log2(r['scales_mapped'])
        log2_m = np.log2(np.maximum(r['mod_mapped'], 1e-30))
        valid = r['mod_mapped'] > 1e-20
        lo = log2_s[valid].min() + 1
        hi = log2_s[valid].max() - 0.5
        fm = valid & (log2_s > lo) & (log2_s < hi)
        hlist.append(np.polyfit(log2_s[fm], log2_m[fm], 1)[0] if fm.sum() >= 3 else np.nan)

h_true_arr = np.array(h_true_arr)
h_meas_g = np.array(h_meas_g)
h_meas_m = np.array(h_meas_m)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for col, (wn, hm) in enumerate([('gaussian', h_meas_g), ('mexican', h_meas_m)]):
    ax = axes[col]
    valid = np.isfinite(hm)
    ax.plot(h_true_arr[valid], hm[valid], 'ko', ms=10)

    if valid.sum() >= 2:
        a_fit, b_fit = np.polyfit(h_true_arr[valid], hm[valid], 1)
        hr = np.array([-0.5, 2.0])
        ax.plot(hr, a_fit * hr + b_fit, 'r-', lw=2,
                label=f'fit: {a_fit:.3f} * h + {b_fit:.3f}')
        ax.plot(hr, hr, 'k--', lw=1, alpha=0.3, label='ideal 1:1')

        for i in range(len(h_true_arr)):
            if valid[i]:
                ax.annotate(f'{h_true_arr[i]:.1f}', xy=(h_true_arr[i], hm[i]),
                            xytext=(5, 5), textcoords='offset points', fontsize=9)

    ax.set_xlabel('h_true'); ax.set_ylabel('h_measured')
    ax.set_title(f'{wn}: a={a_fit:.3f}, b={b_fit:.3f}')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.2)
    ax.set_aspect('equal')

plt.suptitle('DIAGNOSIS: h_meas vs h_true\na!=1 = wrong scale power | b!=0 = vanishing moment offset',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Interpretation
print('\n' + '='*60)
for wn, hm in [('gaussian', h_meas_g), ('mexican', h_meas_m)]:
    valid = np.isfinite(hm)
    if valid.sum() < 2: continue
    a_fit, b_fit = np.polyfit(h_true_arr[valid], hm[valid], 1)
    print(f'\n  {wn}:  h_meas = {a_fit:.4f} * h_true + {b_fit:.4f}')
    if abs(a_fit - 1.0) > 0.05:
        print(f'    MULTIPLICATIVE ERROR: slope {a_fit:.3f} != 1')
    if abs(b_fit) > 0.05:
        print(f'    ADDITIVE ERROR: offset {b_fit:.3f} != 0')
        if abs(b_fit - 1.0) < 0.15:
            print(f'    -> likely 1 extra vanishing moment (factor of a)')
        if abs(b_fit - 2.0) < 0.15:
            print(f'    -> likely 2 extra vanishing moments (factor of a^2)')
    if abs(a_fit - 1.0) < 0.05 and abs(b_fit) < 0.05:
        print(f'    CWT NORMALIZATION IS CORRECT')
        print(f'    If pipeline h is wrong, problem is in chaining/PF')
print('='*60)